# NYC Yellow Taxi — dataset exploration

Exploration that produced the design decisions in `src/ledger/engine/bootstrap.sql`.
Three things this notebook establishes, each of which changed the implementation:

1. **The schema drifts across months.** `cbd_congestion_fee` does not exist before
   2025-01, and `Airport_fee` has drifted in casing. This is why the view uses
   `union_by_name` and `COALESCE(TRY_CAST(...))` rather than a positional union.
2. **The tails are absurd.** Negative fares (refunds) and 200-mile trips are real
   rows. This is why `distribution` clips by default — an unclipped histogram is
   one bin holding every row, which is a wrong answer dressed as a chart.
3. **Congestion pricing began 5 January 2025**, inside the loaded window, which is
   what makes the "after the fare change" question answerable.

Run against the real download (`make fetch`) or, with no arguments, the
deterministic test fixture.


In [1]:
from pathlib import Path

import duckdb

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REAL, FIXTURE = REPO / "data" / "raw", REPO / "tests" / "fixtures" / "data" / "raw"
RAW = REAL if list(REAL.glob("*.parquet")) else FIXTURE
print("reading from:", RAW.relative_to(REPO))

con = duckdb.connect(":memory:")
con.execute("SET VARIABLE trips_glob = ?", [str(RAW / "yellow_tripdata_*.parquet")])
con.execute("SET VARIABLE zones_path = ?", [str(RAW / "taxi_zone_lookup.csv")])
con.execute((REPO / "src" / "ledger" / "engine" / "bootstrap.sql").read_text())
con.sql("SELECT count(*) AS trips FROM ledger.trips")

reading from: tests/fixtures/data/raw


┌───────┐
│ trips │
│ int64 │
├───────┤
│ 45000 │
└───────┘

## 1. Schema drift across months

Each file's own column list, straight from the parquet metadata.

In [2]:
for f in sorted(RAW.glob("yellow_tripdata_*.parquet")):
    cols = {r[0] for r in duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{f}')").fetchall()}
    print(f"{f.name:38} {len(cols):2d} cols  cbd_congestion_fee={'cbd_congestion_fee' in cols}")

yellow_tripdata_2024-12.parquet        19 cols  cbd_congestion_fee=False
yellow_tripdata_2025-01.parquet        20 cols  cbd_congestion_fee=True
yellow_tripdata_2025-02.parquet        20 cols  cbd_congestion_fee=True


The consequence: `union_by_name` is load-bearing. Without it, a positional union
misaligns columns *silently* — no error, just wrong numbers.

In [3]:
con.sql("""
    SELECT strftime(date_trunc('month', pickup_at), '%Y-%m') AS month,
           count(*)                    AS trips,
           count(cbd_congestion_fee)   AS rows_with_cbd_fee,
           round(avg(cbd_congestion_fee), 4) AS avg_cbd_fee
    FROM ledger.trips GROUP BY 1 ORDER BY 1
""")

┌─────────┬───────┬───────────────────┬─────────────┐
│  month  │ trips │ rows_with_cbd_fee │ avg_cbd_fee │
│ varchar │ int64 │       int64       │   double    │
├─────────┼───────┼───────────────────┼─────────────┤
│ 2024-12 │ 15000 │                 0 │        NULL │
│ 2025-01 │ 15000 │             15000 │      0.6509 │
│ 2025-02 │ 15000 │             15000 │        0.75 │
└─────────┴───────┴───────────────────┴─────────────┘

## 2. Distribution tails

Why `distribution` clips and why `top_n` needs `min_group_rows`.

In [4]:
con.sql("""
    SELECT round(min(fare_amount), 2)  AS min_fare,
           round(quantile_cont(fare_amount, 0.50), 2) AS p50,
           round(quantile_cont(fare_amount, 0.99), 2) AS p99,
           round(max(fare_amount), 2)  AS max_fare,
           round(max(trip_distance), 1) AS max_miles
    FROM ledger.trips
""")

┌──────────┬────────┬────────┬──────────┬───────────┐
│ min_fare │  p50   │  p99   │ max_fare │ max_miles │
│  double  │ double │ double │  double  │  double   │
├──────────┼────────┼────────┼──────────┼───────────┤
│    -12.5 │  19.78 │  66.99 │   998.25 │     210.4 │
└──────────┴────────┴────────┴──────────┴───────────┘

A single outlier is enough to win a naive "highest average fare by zone" query.
This is the plausible-but-wrong answer `min_group_rows` exists to prevent —
compare the two rankings below.

In [5]:
print("unguarded — an outlier can win:")
display(
    con.sql("""
    SELECT pickup_zone, count(*) AS trips, round(avg(fare_amount), 2) AS avg_fare
    FROM ledger.trips GROUP BY 1 ORDER BY avg_fare DESC LIMIT 5
""")
)

print("guarded with min_group_rows = 200:")
display(
    con.sql("""
    SELECT pickup_zone, count(*) AS trips, round(avg(fare_amount), 2) AS avg_fare
    FROM ledger.trips GROUP BY 1 HAVING count(*) >= 200
    ORDER BY avg_fare DESC LIMIT 5
""")
)

unguarded — an outlier can win:


┌───────────────────┬───────┬──────────┐
│    pickup_zone    │ trips │ avg_fare │
│      varchar      │ int64 │  double  │
├───────────────────┼───────┼──────────┤
│ Garment District  │  1533 │    23.92 │
│ Park Slope        │  1517 │    23.41 │
│ SoHo              │  1472 │    23.31 │
│ Battery Park City │  1523 │    23.24 │
│ Bloomingdale      │  1419 │    23.07 │
└───────────────────┴───────┴──────────┘

guarded with min_group_rows = 200:


┌───────────────────┬───────┬──────────┐
│    pickup_zone    │ trips │ avg_fare │
│      varchar      │ int64 │  double  │
├───────────────────┼───────┼──────────┤
│ Garment District  │  1533 │    23.92 │
│ Park Slope        │  1517 │    23.41 │
│ SoHo              │  1472 │    23.31 │
│ Battery Park City │  1523 │    23.24 │
│ Bloomingdale      │  1419 │    23.07 │
└───────────────────┴───────┴──────────┘

## 3. Congestion pricing, 5 January 2025

The boundary that makes the brief's example question answerable.

In [6]:
con.sql("""
    SELECT pickup_at >= TIMESTAMP '2025-01-05' AS after_charge,
           count(*)                          AS trips,
           round(avg(total_amount), 2)       AS avg_total,
           round(avg(cbd_congestion_fee), 3) AS avg_cbd_fee
    FROM ledger.trips
    WHERE pickup_at >= TIMESTAMP '2025-01-01' AND pickup_at < TIMESTAMP '2025-02-01'
    GROUP BY 1 ORDER BY 1
""")

┌──────────────┬───────┬───────────┬─────────────┐
│ after_charge │ trips │ avg_total │ avg_cbd_fee │
│   boolean    │ int64 │  double   │   double    │
├──────────────┼───────┼───────────┼─────────────┤
│ false        │  1983 │     33.15 │         0.0 │
│ true         │ 13017 │     33.61 │        0.75 │
└──────────────┴───────┴───────────┴─────────────┘

## 4. Cardinality — what can safely be grouped

The numbers behind the `cardinality_exceeded` guard: the catalogue caches these
so a hopeless `group_by` is refused without touching the engine.

In [7]:
con.sql("""
    SELECT 'pickup_at' AS column, approx_count_distinct(pickup_at) AS approx_distinct FROM ledger.trips
    UNION ALL SELECT 'pickup_zone',    approx_count_distinct(pickup_zone)    FROM ledger.trips
    UNION ALL SELECT 'pickup_borough', approx_count_distinct(pickup_borough) FROM ledger.trips
    UNION ALL SELECT 'payment_type',   approx_count_distinct(payment_type)   FROM ledger.trips
    ORDER BY approx_distinct DESC
""")

┌────────────────┬─────────────────┐
│     column     │ approx_distinct │
│    varchar     │      int64      │
├────────────────┼─────────────────┤
│ pickup_at      │           46236 │
│ pickup_zone    │              32 │
│ payment_type   │               4 │
│ pickup_borough │               3 │
└────────────────┴─────────────────┘

## 5. Nulls

What the catalogue's `null_fraction` will report, and why `passenger_count`
needs a caveat in its description.

In [8]:
con.sql("""
    SELECT 'passenger_count' AS column,
           round(100.0 * count(*) FILTER (passenger_count IS NULL) / count(*), 2) AS null_pct
    FROM ledger.trips
    UNION ALL SELECT 'pickup_zone',
           round(100.0 * count(*) FILTER (pickup_zone IS NULL) / count(*), 2) FROM ledger.trips
    UNION ALL SELECT 'cbd_congestion_fee',
           round(100.0 * count(*) FILTER (cbd_congestion_fee IS NULL) / count(*), 2) FROM ledger.trips
    ORDER BY null_pct DESC
""")

┌────────────────────┬──────────┐
│       column       │ null_pct │
│      varchar       │  double  │
├────────────────────┼──────────┤
│ cbd_congestion_fee │    33.33 │
│ passenger_count    │     1.98 │
│ pickup_zone        │      0.0 │
└────────────────────┴──────────┘